# M02 lab solution: generator choice

**Brief.** Answer 20 questions with a fast model, a reasoning model at two effort levels and an open model, and record accuracy, latency and cost per answer.

Each question comes with its golden evidence spans as context, so the test measures the generator alone, not retrieval. The models follow the module's notebooks: `gpt-4o-mini` (fast) and `o4-mini` (reasoning, `reasoning_effort` low and high) through the OpenAI SDK, as in `03a` and `04a`, and `meta-llama/Llama-3.1-8B-Instruct` through the Hugging Face Inference API, as in `02a`. A separate judge model grades each answer against `reference_answer`.

Keys: `OPENAI_API_KEY`, `HF_TOKEN`, `EXPERIENTIALLABS_API_KEY` (judge).

In [ ]:
import sys
from pathlib import Path

from dotenv import load_dotenv

sys.path.insert(0, str(Path("../../../../src").resolve()))
load_dotenv()

DATA = Path("../data")

import os
import time

import pandas as pd
from huggingface_hub import InferenceClient
from openai import OpenAI

from helpers.utils import get_experientiallabs_llm
from lab_kit import load_jsonl

## 1. The 20 questions

The golden set has 15 `lookup` and 4 `numeric` questions. The 20th is `g02`, the first `multi_hop` question, so the set reaches the brief's 20.

In [ ]:
golden = load_jsonl(DATA / "larkspur" / "golden.jsonl")
questions = [g for g in golden if g["type"] in ("lookup", "numeric")]
questions.append(next(g for g in golden if g["id"] == "g02"))
print(len(questions), "questions:", ", ".join(g["id"] for g in questions))

## 2. Prices

Cost is computed from `usage`, never from the reply text: reasoning tokens are billed at the output rate and never returned. The values below are the ones in notebook `03a`; check them on the provider's pricing page and set `PRICES_CHECKED_ON` before the run. The open model's price depends on the provider Hugging Face routes to, so it stays `None` (cost shown as n/a) unless you fill it in.

In [ ]:
PRICES_CHECKED_ON = None  # e.g. "2026-11-08", the day you read the pricing page
PRICES = {  # USD per 1M tokens: (input, output)
    "gpt-4o-mini": (0.15, 0.60),
    "o4-mini": (1.10, 4.40),
    "meta-llama/Llama-3.1-8B-Instruct": None,
}
if PRICES_CHECKED_ON is None:
    print("Warning: prices not checked against the provider pages; set PRICES_CHECKED_ON.")

## 3. One call function per provider

Each returns the answer, the seconds it took and the token counts from `usage`.

In [ ]:
openai_client = OpenAI()
hf_client = InferenceClient(api_key=os.environ["HF_TOKEN"])

SYSTEM = ("Answer the customer's question using only the context. "
          "Answer in one or two sentences. If the context does not answer it, say you don't know.")

def user_prompt(g: dict) -> str:
    context = "\n".join(f"- {s}" for s in g["evidence_spans"])
    return f"Context:\n{context}\n\nQuestion: {g['question']}"

def call_openai(model: str, g: dict, effort: str | None = None) -> dict:
    kwargs = {"reasoning_effort": effort} if effort else {"temperature": 0}
    start = time.perf_counter()
    resp = openai_client.chat.completions.create(
        model=model,
        messages=[{"role": "system", "content": SYSTEM}, {"role": "user", "content": user_prompt(g)}],
        **kwargs)
    seconds = time.perf_counter() - start
    details = getattr(resp.usage, "completion_tokens_details", None)
    return {"answer": (resp.choices[0].message.content or "").strip(), "seconds": seconds,
            "prompt_tokens": resp.usage.prompt_tokens, "completion_tokens": resp.usage.completion_tokens,
            "reasoning_tokens": getattr(details, "reasoning_tokens", 0) or 0}

def call_hf(model: str, g: dict) -> dict:
    start = time.perf_counter()
    resp = hf_client.chat_completion(
        messages=[{"role": "system", "content": SYSTEM}, {"role": "user", "content": user_prompt(g)}],
        model=model, max_tokens=300, temperature=0)
    seconds = time.perf_counter() - start
    usage = resp.usage
    return {"answer": (resp.choices[0].message.content or "").strip(), "seconds": seconds,
            "prompt_tokens": getattr(usage, "prompt_tokens", None),
            "completion_tokens": getattr(usage, "completion_tokens", None), "reasoning_tokens": 0}

def cost_usd(model: str, r: dict) -> float | None:
    price = PRICES.get(model)
    if price is None or r["prompt_tokens"] is None:
        return None
    return (r["prompt_tokens"] * price[0] + r["completion_tokens"] * price[1]) / 1_000_000

CONFIGS = [  # label, provider, model, reasoning effort
    ("fast", "openai", "gpt-4o-mini", None),
    ("reasoning-low", "openai", "o4-mini", "low"),
    ("reasoning-high", "openai", "o4-mini", "high"),
    ("open", "hf", "meta-llama/Llama-3.1-8B-Instruct", None),
]

## 4. A judge for accuracy

The judge sees the question, the reference answer and the candidate answer, and replies `CORRECT` or `INCORRECT`. It is a different model from the four under test, so no model grades its own answers. Read a few verdicts by hand before trusting the column.

In [ ]:
judge = get_experientiallabs_llm()

JUDGE_PROMPT = """You grade answers to customer questions about a hosting company.
Question: {question}
Reference answer: {reference}
Candidate answer: {candidate}

Is the candidate answer correct and complete with respect to the reference? Extra wording is fine;
a missing or wrong fact is not. Reply with exactly one word: CORRECT or INCORRECT."""

def is_correct(g: dict, answer: str) -> bool:
    verdict = judge.invoke(JUDGE_PROMPT.format(question=g["question"], reference=g["reference_answer"],
                                               candidate=answer)).content.strip().upper()
    return verdict.startswith("CORRECT")

## 5. Run every question through every configuration

In [ ]:
rows = []
for label, provider, model, effort in CONFIGS:
    for g in questions:
        r = call_openai(model, g, effort) if provider == "openai" else call_hf(model, g)
        rows.append({"config": label, "model": model, "id": g["id"], **r,
                     "cost_usd": cost_usd(model, r), "correct": is_correct(g, r["answer"])})
    print(f"{label:<15} done")

runs = pd.DataFrame(rows)
runs[["config", "id", "correct", "seconds", "reasoning_tokens", "answer"]].head(10)

## Results

In [ ]:
summary = (runs.groupby(["config", "model"], sort=False)
                .agg(questions=("id", "size"), accuracy=("correct", "mean"),
                     mean_seconds=("seconds", "mean"), mean_reasoning_tokens=("reasoning_tokens", "mean"),
                     cost_per_answer_usd=("cost_usd", lambda s: s.mean() if s.notna().all() else None))
                .reset_index())
print("Prices checked on:", PRICES_CHECKED_ON)
summary

In [ ]:
# Answers the judge marked wrong, to audit the judge as well as the models.
runs.loc[~runs["correct"], ["config", "id", "answer"]]

## What this shows

- With the evidence in the prompt, the questions test reading, not knowledge, so a fast model may match the reasoning model at a fraction of the cost. The table says whether it did on this set.
- `reasoning_effort` moves the reasoning token count and the latency. Compare the two `o4-mini` rows: a step that adds tokens without adding accuracy is pure cost.
- The open model's row shows what the hosted call costs in latency; its price depends on the provider Hugging Face routes to.